# SPACESHIP TITANIC

In [1]:
import copy
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression


## Data Import

In [2]:
train_data = pd.read_csv("./data/train.csv")
test_data = pd.read_csv("./data/test.csv")

## Data Preprocess
- Transported/CryoSleep 轉成 0/1
- PassengerId、Name、Cabin 不直接當特徵
- Cabin 缺失值補上 Missing//Missing
    - 把 Cabin 拆成 CabinDeck、CabinNum、CabinSide


In [3]:
def to_binary(series):
    return series.astype(str).map({"False": 0, "True": 1}).astype(float)


def add_basic_features(df):
    df = df.copy()
    df = df.drop(columns=["PassengerId", "Name"], errors="ignore")
    cabin = df["Cabin"].fillna("Missing//Missing").str.split("/", expand=True)
    df["CabinDeck"] = cabin[0]
    df["CabinNum"] = pd.to_numeric(cabin[1], errors="coerce")
    df["CabinSide"] = cabin[2]
    return df.drop(columns=["Cabin"])


# Split train / validation first.
X = train_data.drop(columns=["Transported"]).copy()
y = to_binary(train_data["Transported"])
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train = add_basic_features(X_train)
X_val = add_basic_features(X_val)


## Baseline 0
先看一個最簡單的 `Transported = CryoSleep` validation accuracy
- 把 validation set 裡 CryoSleep 的缺失值，補成算出的眾數

In [4]:
# 先從 training split 的 CryoSleep 找出最常出現的值
cryo_mode = X_train["CryoSleep"].mode(dropna=True)[0]
# 把 validation split 裡 CryoSleep 的缺失值，補成剛剛算出的眾數
cryo_val = X_val["CryoSleep"].astype("string").fillna(cryo_mode)
cryo_pred = to_binary(cryo_val)
# 直接算 accuracy
acc_b0 = accuracy_score(y_val, cryo_pred >= 0.5)
print(f"Baseline 0 | Transported = CryoSleep | Acc: {acc_b0:.4f}")

Baseline 0 | Transported = CryoSleep | Acc: 0.7131


## Baseline1
Ridge fit training split，再算 validation accuracy

In [5]:
# Baseline 1: Ridge on train split, evaluated on validation split.
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")), # 數值欄位用 median 補缺失值
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")), # 類別欄位用 most_frequent 補缺失值
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline([
    ("preprocess", preprocess),
    ("ridge", Ridge(alpha=1.0)),
])

model.fit(X_train, y_train)
val_pred = model.predict(X_val)
acc_b1 = accuracy_score(y_val, val_pred >= 0.5)
print(f"Baseline 1 | Ridge val Acc: {acc_b1:.4f}")


Baseline 1 | Ridge val Acc: 0.7596


# XGBoost + Feature Selection + Early Stopping
- 先做缺值補齊 + one-hot
- 用 SelectKBest(f_regression) 選特徵
- 直接用外層 X_val 做 early stopping
- 畫 train / val logloss 曲線看收斂狀況

In [6]:
from xgboost import XGBClassifier



# XGBoost does not need scaling, so we only impute missing values and one-hot encode categoricals.
xgb_numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
xgb_categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

xgb_preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), xgb_numeric_features),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            xgb_categorical_features,
        ),
    ]
)

X_train_pre = xgb_preprocess.fit_transform(X_train)
X_val_pre = xgb_preprocess.transform(X_val)
xgb_feature_names = xgb_preprocess.get_feature_names_out()

# Keep the feature screen compact, then let XGBoost learn on the selected columns.
XGB_SELECT_K = 20
N_ESTIMATORS = 1000
LR=0.05
MAX_DEPTH=3
SUBSAMPLE=0.85
COLSAMPLE_BYTREE=0.85
REG_LAMBDA=1.0
ESR=50


xgb_k = min(XGB_SELECT_K, X_train_pre.shape[1])
selector = SelectKBest(score_func=f_regression, k=xgb_k)
X_train_sel = selector.fit_transform(X_train_pre, y_train)
X_val_sel = selector.transform(X_val_pre)
selected_xgb_feature_names = xgb_feature_names[selector.get_support()]

print(f"XGB_SELECT_K = {xgb_k}")
print(f"preprocessed feature count = {X_train_pre.shape[1]}")
print(f"selected feature count = {len(selected_xgb_feature_names)}")
print("selected features:")
print(list(selected_xgb_feature_names))

xgb_model = XGBClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LR,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    reg_lambda=REG_LAMBDA,
    objective="binary:logistic",
    eval_metric="logloss",
    early_stopping_rounds=ESR,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(
    X_train_sel,
    y_train.astype(int),
    eval_set=[(X_train_sel, y_train.astype(int)), (X_val_sel, y_val.astype(int))],
    verbose=False,
)
train_acc_xgb = accuracy_score(y_train.astype(int), xgb_model.predict(X_train_sel))
val_acc_xgb = accuracy_score(y_val.astype(int), xgb_model.predict(X_val_sel))
print(f"XGBoost | train acc: {train_acc_xgb:.4f} | val acc: {val_acc_xgb:.4f}")
print(f"best_iteration = {xgb_model.best_iteration}")
print(f"best_score = {xgb_model.best_score}")


ModuleNotFoundError: No module named 'xgboost'

## Figures

In [ ]:

evals_result = xgb_model.evals_result()
train_logloss = evals_result["validation_0"]["logloss"]
val_logloss = evals_result["validation_1"]["logloss"]
rounds = np.arange(1, len(train_logloss) + 1)
best_round = (xgb_model.best_iteration + 1) if xgb_model.best_iteration is not None else int(np.argmin(val_logloss) + 1)
best_idx = best_round - 1
best_val_logloss = val_logloss[best_idx]
zoom_start = max(1, best_round - 30)
zoom_mask = rounds >= zoom_start
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rounds, train_logloss, label='Train logloss', linewidth=2)
axes[0].plot(rounds, val_logloss, label='Validation logloss', linewidth=2)
axes[0].axvline(best_round, color='tab:red', linestyle='--', alpha=0.35, label=f'Best round = {best_round}')
axes[0].scatter(best_round, best_val_logloss, color='tab:red', s=35, zorder=3)
axes[0].set_xlabel('Boosting round')
axes[0].set_ylabel('Logloss')
axes[0].set_title('Full curve')
axes[0].legend()

axes[1].plot(rounds[zoom_mask], np.array(train_logloss)[zoom_mask], label='Train logloss', linewidth=2)
axes[1].plot(rounds[zoom_mask], np.array(val_logloss)[zoom_mask], label='Validation logloss', linewidth=2)
axes[1].axvline(best_round, color='tab:red', linestyle='--', alpha=0.35, label=f'Best round = {best_round}')
axes[1].scatter(best_round, best_val_logloss, color='tab:red', s=35, zorder=3)
axes[1].annotate(
    f'best logloss = {best_val_logloss:.4f}',
    xy=(best_round, best_val_logloss),
    xytext=(12, 14),
    textcoords='offset points',
    fontsize=10,
    color='tab:red',
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='tab:red', alpha=0.9),
    arrowprops=dict(arrowstyle='->', color='tab:red', lw=1),
)
axes[1].set_xlabel('Boosting round')
axes[1].set_ylabel('Logloss')
axes[1].set_title(f'Zoomed view from round {zoom_start}')
axes[1].legend()

fig.suptitle(f'Training / Validation Logloss vs. Boosting Round | best logloss = {best_val_logloss:.4f}')
fig.tight_layout()
plt.show()